# Custom Fitness Functions: Build Your Own

Learn to design custom fitness functions that match your specific business needs:

1. Understand fitness function structure
2. Add constraint penalties
3. Combine multiple objectives
4. Debug and validate fitness functions

## Setup

In [1]:
import sys
from pathlib import Path

# Add src to path for importing pso_segmentation
sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd

from pso_segmentation import compute_metrics, segment_scores, validate_cuts

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## Generate Sample Data

In [2]:
n_samples = 2000
scores = np.random.beta(a=2, b=5, size=n_samples)
labels = (np.random.rand(n_samples) < scores).astype(int)

print(f"Dataset: {n_samples} customers")
print(f"Default rate: {labels.mean():.1%}")

Dataset: 2000 customers
Default rate: 28.4%


## 1. Fitness Function Structure

All fitness functions follow this template:

In [3]:
def template_fitness(cuts, scores, labels):
    """
    Template fitness function.

    Parameters
    ----------
    cuts : np.ndarray
        Array of segment boundaries (n_segments - 1 values)
    scores : np.ndarray
        Risk scores (typically 0-1)
    labels : np.ndarray
        Binary labels (0 or 1)

    Returns
    -------
    float
        Fitness value (higher = better)
        Must be in a reasonable range (typically 0-1 or -1 to 1)
    """
    # 1. VALIDATE CUTS
    if not validate_cuts(cuts, scores):
        return 0.0  # Return 0 for invalid cuts

    # 2. COMPUTE METRICS
    result = compute_metrics(cuts, scores, labels)

    # 3. COMPUTE OBJECTIVES
    fitness = result.r2  # Base objective

    # 4. APPLY PENALTIES
    # (optional - customize for your needs)

    # 5. RETURN
    return fitness


print("Template defined!")

Template defined!


## 2. Custom Fitness 1: Strict Monotonicity

Hard constraint: segments must have strictly increasing default rates:

In [4]:
def fitness_strict_monotonic(cuts, scores, labels):
    """Maximize R² but enforce strict monotonicity (no penalty, rejection)."""
    if not validate_cuts(cuts, scores):
        return 0.0

    result = compute_metrics(scores, labels, cuts)
    pd = result.pd_by_segment

    # Check strict monotonicity
    for i in range(len(pd) - 1):
        if pd[i] >= pd[i + 1]:  # Not strictly increasing
            return 0.0  # Reject

    # If monotonic, return R²
    return result.r2


# Test it
result1 = segment_scores(
    scores, labels, lambda cuts: fitness_strict_monotonic(cuts, scores, labels)
)

print("Strict Monotonic Fitness:")
print(f"  R²: {result1.r2:.4f}")
print(f"  PD by segment: {np.round(result1.pd_by_segment, 3)}")
print(
    f"  Is strictly monotonic: {all(result1.pd_by_segment[i] < result1.pd_by_segment[i + 1] for i in range(len(result1.pd_by_segment) - 1))}"
)

Strict Monotonic Fitness:
  R²: 0.1401
  PD by segment: [0.135 0.285 0.449 0.657]
  Is strictly monotonic: True


## 3. Custom Fitness 2: Soft Penalty

Soft constraint with penalty for monotonicity violations:

In [5]:
def fitness_soft_monotonic(cuts, scores, labels, penalty_weight=0.3):
    """Maximize R² with soft monotonicity penalty."""
    if not validate_cuts(cuts, scores):
        return 0.0

    result = compute_metrics(scores, labels, cuts)
    pd = result.pd_by_segment

    # Start with R²
    fitness = result.r2

    # Penalize monotonicity violations
    monotonic_penalty = 0.0
    for i in range(len(pd) - 1):
        if pd[i] > pd[i + 1]:  # Violates monotonicity
            violation = pd[i] - pd[i + 1]
            monotonic_penalty += violation

    # Apply penalty
    fitness -= penalty_weight * monotonic_penalty
    return max(0.0, fitness)  # Ensure non-negative


# Create a wrapper to fix the parameters
def fitness_soft_wrapper(cuts):
    return fitness_soft_monotonic(cuts, scores, labels, penalty_weight=0.4)


result2 = segment_scores(scores, labels, fitness_soft_wrapper)

print("Soft Monotonic Fitness (weight=0.4):")
print(f"  R²: {result2.r2:.4f}")
print(f"  PD by segment: {np.round(result2.pd_by_segment, 3)}")

Soft Monotonic Fitness (weight=0.4):
  R²: 0.1407
  PD by segment: [0.135 0.284 0.396 0.603]


## 4. Custom Fitness 3: Multi-Objective

Combine R² with segment balance and homogeneity:

In [6]:
def fitness_multi_objective(cuts, scores, labels, w_r2=0.5, w_balance=0.3, w_homo=0.2):
    """
    Multi-objective: R² + Balance + Homogeneity.

    Parameters
    ----------
    w_r2 : float
        Weight for R² (explained variance)
    w_balance : float
        Weight for segment balance
    w_homo : float
        Weight for within-segment homogeneity
    """
    if not validate_cuts(cuts, scores):
        return 0.0

    result = compute_metrics(scores, labels, cuts)

    # Component 1: R²
    r2_component = result.r2  # Already 0-1

    # Component 2: Balance (penalize unequal sizes)
    proportions = result.segment_proportions
    expected_prop = 1.0 / len(proportions)
    balance_penalty = np.mean(np.abs(proportions - expected_prop))
    balance_component = 1.0 - balance_penalty  # Convert to "higher is better"

    # Component 3: Homogeneity
    homo_component = result.h_intra  # Already 0-1 ideally

    # Weighted sum
    fitness = w_r2 * r2_component + w_balance * balance_component + w_homo * homo_component

    return fitness


def fitness_multi_wrapper(cuts):
    return fitness_multi_objective(cuts, scores, labels, w_r2=0.5, w_balance=0.3, w_homo=0.2)


result3 = segment_scores(scores, labels, fitness_multi_wrapper)

print("Multi-Objective Fitness (R²=0.5, Balance=0.3, Homo=0.2):")
print(f"  R²: {result3.r2:.4f}")
print(f"  H_intra: {result3.h_intra:.4f}")
print(f"  Proportions: {np.round(result3.segment_proportions, 3)}")

Multi-Objective Fitness (R²=0.5, Balance=0.3, Homo=0.2):
  R²: 0.0007
  H_intra: 406.8526
  Proportions: [0.    0.997 0.002]


## 5. Custom Fitness 4: Business-Specific

Optimize for credit portfolio management (minimize concentration risk):

In [7]:
def fitness_concentration_risk(cuts, scores, labels):
    """
    Optimize for portfolio management.

    Goals:
    1. Maximize R² (predictive power)
    2. Minimize concentration in high-risk segment
    3. Maintain minimum size for each segment
    """
    if not validate_cuts(cuts, scores):
        return 0.0

    result = compute_metrics(scores, labels, cuts)

    # Base: R²
    fitness = result.r2

    # Penalize high-risk segment concentration
    # (Last segment has highest risk)
    risk_concentration = result.segment_proportions[-1]
    if risk_concentration > 0.4:  # If >40% in high-risk
        concentration_penalty = (risk_concentration - 0.4) * 0.5
        fitness -= concentration_penalty

    # Penalize small segments (unstable)
    min_size = result.segment_proportions.min()
    if min_size < 0.05:  # If segment <5%
        size_penalty = (0.05 - min_size) * 0.3
        fitness -= size_penalty

    return max(0.0, fitness)


result4 = segment_scores(
    scores, labels, lambda cuts: fitness_concentration_risk(cuts, scores, labels)
)

print("Portfolio-Focused Fitness:")
print(f"  R²: {result4.r2:.4f}")
print(f"  Risk concentration (last segment): {result4.segment_proportions[-1]:.1%}")
print(f"  Min segment size: {result4.segment_proportions.min():.1%}")

Portfolio-Focused Fitness:
  R²: 0.1402
  Risk concentration (last segment): 9.5%
  Min segment size: 9.5%


## 6. Comparison of All Fitness Functions

In [8]:
# Compare all custom fitness functions
comparison = pd.DataFrame(
    {
        "Fitness Type": [
            "Strict Monotonic",
            "Soft Monotonic",
            "Multi-Objective",
            "Portfolio-Focused",
        ],
        "R²": [
            result1.r2,
            result2.r2,
            result3.r2,
            result4.r2,
        ],
        "Is Monotonic": [
            all(
                result1.pd_by_segment[i] < result1.pd_by_segment[i + 1]
                for i in range(len(result1.pd_by_segment) - 1)
            ),
            all(
                result2.pd_by_segment[i] <= result2.pd_by_segment[i + 1]
                for i in range(len(result2.pd_by_segment) - 1)
            ),
            all(
                result3.pd_by_segment[i] <= result3.pd_by_segment[i + 1]
                for i in range(len(result3.pd_by_segment) - 1)
            ),
            all(
                result4.pd_by_segment[i] <= result4.pd_by_segment[i + 1]
                for i in range(len(result4.pd_by_segment) - 1)
            ),
        ],
        "Min Prop": [
            result1.segment_proportions.min(),
            result2.segment_proportions.min(),
            result3.segment_proportions.min(),
            result4.segment_proportions.min(),
        ],
        "Risk Conc": [
            result1.segment_proportions[-1],
            result2.segment_proportions[-1],
            result3.segment_proportions[-1],
            result4.segment_proportions[-1],
        ],
    }
)

print("Comparison of Custom Fitness Functions:")
print(comparison.to_string(index=False))

Comparison of Custom Fitness Functions:
     Fitness Type       R²  Is Monotonic  Min Prop  Risk Conc
 Strict Monotonic 0.140101          True    0.0990     0.0990
   Soft Monotonic 0.140684          True    0.1135     0.1700
  Multi-Objective 0.000656          True    0.0005     0.0020
Portfolio-Focused 0.140180          True    0.0955     0.0955


## 7. Debugging Your Fitness Function

Techniques for testing and debugging:

In [9]:
def fitness_debug(cuts, scores, labels, verbose=True):
    """Fitness function with debugging output."""
    if verbose:
        print(f"\nEvaluating cuts: {np.round(cuts, 3)}")

    # Validate
    if not validate_cuts(cuts, scores):
        if verbose:
            print("  ✗ Invalid cuts (not sorted or out of range)")
        return 0.0
    else:
        if verbose:
            print("  ✓ Cuts valid")

    # Compute metrics
    result = compute_metrics(scores, labels, cuts)
    if verbose:
        print(f"  R²: {result.r2:.4f}")
        print(f"  PD: {np.round(result.pd_by_segment, 3)}")
        print(f"  Props: {np.round(result.segment_proportions, 3)}")

    # Compute fitness
    fitness = result.r2
    if verbose:
        print(f"  Fitness: {fitness:.4f}")

    return fitness


# Test with manual cuts
test_cuts = np.array([0.3, 0.6])
print("Testing fitness function with manual cuts:")
fitness_val = fitness_debug(test_cuts, scores, labels, verbose=True)

Testing fitness function with manual cuts:

Evaluating cuts: [0.3 0.6]
  ✓ Cuts valid
  R²: 0.1044
  PD: [0.166 0.41  0.714]
  Props: [0.562 0.399 0.038]
  Fitness: 0.1044


## 8. Best Practices

Guidelines for writing good fitness functions:

In [10]:
print("""
BEST PRACTICES FOR FITNESS FUNCTIONS:

1. INPUT VALIDATION
   ✓ Always check validate_cuts(cuts, scores) first
   ✓ Return 0.0 for invalid cuts
   ✓ Handle edge cases (empty segments, NaN values)

2. OBJECTIVE DESIGN
   ✓ Higher fitness = better solution
   ✓ Keep fitness in reasonable range (0-1 or -1 to 1)
   ✓ Base objectives should be normalized (e.g., R² is already 0-1)

3. CONSTRAINT PENALTIES
   ✓ Use soft penalties instead of hard rejections (usually)
   ✓ Scale penalties to match objective magnitude
   ✓ Example: if base fitness is 0-1, penalties should be 0-0.3

4. MULTI-OBJECTIVE WEIGHTS
   ✓ Ensure weights sum to 1.0 (or normalize afterwards)
   ✓ Test different weight combinations
   ✓ Document what each weight controls

5. PERFORMANCE
   ✓ Fitness function is called 1000s of times
   ✓ Keep computation lightweight
   ✓ Avoid complex loops; use NumPy vectorization

6. TESTING
   ✓ Test with known cuts (e.g., equal percentiles)
   ✓ Verify constraints are actually enforced
   ✓ Check edge cases (min/max scores)
   ✓ Use verbose mode for debugging

7. DOCUMENTATION
   ✓ Document parameters (especially weights)
   ✓ Explain what each penalty does
   ✓ Include example usage
""")


BEST PRACTICES FOR FITNESS FUNCTIONS:

1. INPUT VALIDATION
   ✓ Always check validate_cuts(cuts, scores) first
   ✓ Return 0.0 for invalid cuts
   ✓ Handle edge cases (empty segments, NaN values)

2. OBJECTIVE DESIGN
   ✓ Higher fitness = better solution
   ✓ Keep fitness in reasonable range (0-1 or -1 to 1)
   ✓ Base objectives should be normalized (e.g., R² is already 0-1)

3. CONSTRAINT PENALTIES
   ✓ Use soft penalties instead of hard rejections (usually)
   ✓ Scale penalties to match objective magnitude
   ✓ Example: if base fitness is 0-1, penalties should be 0-0.3

4. MULTI-OBJECTIVE WEIGHTS
   ✓ Ensure weights sum to 1.0 (or normalize afterwards)
   ✓ Test different weight combinations
   ✓ Document what each weight controls

5. PERFORMANCE
   ✓ Fitness function is called 1000s of times
   ✓ Keep computation lightweight
   ✓ Avoid complex loops; use NumPy vectorization

6. TESTING
   ✓ Test with known cuts (e.g., equal percentiles)
   ✓ Verify constraints are actually enforced

## Key Takeaways

✅ **Fitness Function Structure:**
- Validate cuts
- Compute metrics
- Calculate objectives
- Apply penalties
- Return float value

✅ **Constraint Types:**
- Hard constraints: Return 0 if violated
- Soft constraints: Reduce fitness score

✅ **Multi-Objective:**
- Normalize each objective to 0-1
- Use weights to balance objectives
- Test different weights

✅ **Business Metrics:**
- Can optimize for any metric
- Combine multiple metrics
- Weight by business priority

## Next Steps

👉 **04_business_use_case.ipynb** - Apply custom fitness to real business scenario